In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    BaggingClassifier,
    AdaBoostClassifier,
    StackingClassifier
)

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

np.random.seed(42)

data_size = 200

df = pd.DataFrame({
    'Age': np.random.randint(18, 60, data_size),
    'Income': np.random.randint(20000, 120000, data_size),
    'Browsing_Time': np.random.randint(1, 20, data_size),
    'Pages_Visited': np.random.randint(1, 50, data_size),
    'Purchased': np.random.randint(0, 2, data_size)
})

print("First 5 Rows:\n")

print(df.head())

X = df.drop('Purchased', axis=1)

y = df['Purchased']

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

def evaluate_model(name, model):

    y_pred = model.predict(X_test)

    print("\n====================================")

    print(f"{name} Results")

    print("====================================")

    print("\nConfusion Matrix:\n")

    print(confusion_matrix(y_test, y_pred))

    print("\nAccuracy :", accuracy_score(y_test, y_pred))

    print("Precision:", precision_score(y_test, y_pred))

    print("Recall   :", recall_score(y_test, y_pred))

    print("F1 Score :", f1_score(y_test, y_pred))

    print("\nClassification Report:\n")

    print(classification_report(y_test, y_pred))

bagging_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    random_state=42
)

bagging_model.fit(X_train, y_train)

evaluate_model("Bagging Decision Tree", bagging_model)

adaboost_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

adaboost_model.fit(X_train, y_train)

evaluate_model("AdaBoost Classifier", adaboost_model)

base_models = [
    ('dt', DecisionTreeClassifier(random_state=42)),
    ('svm', SVC(probability=True, random_state=42)),
    ('lr', LogisticRegression())
]

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression()
)

stacking_model.fit(X_train, y_train)

evaluate_model("Stacking Classifier", stacking_model)

models = {
    "Bagging": bagging_model,
    "AdaBoost": adaboost_model,
    "Stacking": stacking_model
}

print("\n====================================")

print("Model Accuracy Comparison")

print("====================================")

for name, model in models.items():

    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)

    print(f"{name} Accuracy: {acc:.4f}")